### Nexmark

In [ ]:
from pyflink.table import EnvironmentSettings, TableEnvironment
import os

from pathlib import Path

from pyflink.java_gateway import get_gateway
from pyflink.datastream import StreamExecutionEnvironment
from pyflink.table import StreamTableEnvironment, ExplainDetail

gateway = get_gateway()
string_class = gateway.jvm.String
string_array = gateway.new_array(string_class, 0)
stream_env = gateway.jvm.org.apache.flink.streaming.api.environment.StreamExecutionEnvironment
j_stream_exection_environment = stream_env.createRemoteEnvironment(
    "localhost", 
    8081, 
    string_array
)

env = StreamExecutionEnvironment(j_stream_exection_environment)
env.set_parallelism(4)
table_env = StreamTableEnvironment.create(env)
jar_path = Path("../flink-sql-connector-kafka-4.0.0-2.0.jar").resolve().as_uri()
table_env.get_config().set("pipeline.jars", jar_path)
t_env=table_env
current_dir = os.getcwd()
t_env.get_config().get_configuration().set_string("table.plan.force-recompile", "true")
# Define the source table using DDL (update the file path as needed)
source_ddl = """
CREATE TABLE kafka (
    event_type int,
    person ROW<
        id  BIGINT,
        name  VARCHAR,
        emailAddress  VARCHAR,
        creditCard  VARCHAR,
        city  VARCHAR,
        state  VARCHAR,
        `dateTime` TIMESTAMP(3),
        extra  VARCHAR>,
    auction ROW<
        id  BIGINT,
        itemName  VARCHAR,
        description  VARCHAR,
        initialBid  BIGINT,
        reserve  BIGINT,
        `dateTime`  TIMESTAMP(3),
        expires  TIMESTAMP(3),
        seller  BIGINT,
        category  BIGINT,
        extra  VARCHAR>,
    bid ROW<
        auction  BIGINT,
        bidder  BIGINT,
        price  BIGINT,
        channel  VARCHAR,
        url  VARCHAR,
        `dateTime`  TIMESTAMP(3),
        extra  VARCHAR>,
    `dateTime` AS
        CASE
            WHEN event_type = 0 THEN person.`dateTime`
            WHEN event_type = 1 THEN auction.`dateTime`
            ELSE bid.`dateTime`
        END,
    WATERMARK FOR `dateTime` AS `dateTime` - INTERVAL '4' SECOND
) WITH (
    'connector' = 'kafka',
    'topic' = 'event-demo',
    'properties.bootstrap.servers' = 'kafka-service.kafka.svc.cluster.local:9092',
    'properties.group.id' = 'nexmark',
    'scan.startup.mode' = 'earliest-offset',
    'sink.partitioner' = 'round-robin',
    'format' = 'json'
);
"""
t_env.execute_sql(source_ddl)

# Define the sink table using DDL with the print connector for debugging/output
sink_ddl = """
CREATE TABLE nexmark_q5 (
  auction  BIGINT,
  num  BIGINT
) WITH (
  'connector' = 'print'
);
"""
t_env.execute_sql(sink_ddl)

query_q5= """
INSERT INTO nexmark_q5
SELECT
    AuctionBids.auction,
    AuctionBids.num
FROM (
    SELECT
        bid.auction,
        COUNT(*) AS num,
        HOP_START(`dateTime`, INTERVAL '2' SECOND, INTERVAL '10' SECOND) as starttime,
        HOP_END(`dateTime`, INTERVAL '2' SECOND, INTERVAL '10' SECOND) as endtime
    FROM kafka
    GROUP BY
        bid.auction,
        HOP(`dateTime`, INTERVAL '2' SECOND, INTERVAL '10' SECOND)
) AS AuctionBids
JOIN (
    SELECT
        MAX(CountBids.num) as max_num,
        CountBids.starttime,
        CountBids.endtime
    FROM (
        SELECT
            bid.auction,
            COUNT(*) AS num,
            HOP_START(`dateTime`, INTERVAL '2' SECOND, INTERVAL '10' SECOND) as starttime,
            HOP_END(`dateTime`, INTERVAL '2' SECOND, INTERVAL '10' SECOND) as endtime
        FROM kafka
        WHERE bid IS NOT NULL
        GROUP BY
            bid.auction,
            HOP(`dateTime`, INTERVAL '2' SECOND, INTERVAL '10' SECOND)
    ) AS CountBids
    GROUP BY CountBids.starttime, CountBids.endtime
) AS MaxBids
ON
    AuctionBids.starttime = MaxBids.starttime AND
    AuctionBids.endtime = MaxBids.endtime AND
    AuctionBids.num = MaxBids.max_num;
"""

# Exécuter la requête pour lancer le job
t_env.execute_sql(query_q5)

Py4JJavaError: An error occurred while calling o10.executeSql.
: org.apache.flink.table.api.ValidationException: Unable to create a source for reading table 'default_catalog.default_database.kafka'.

Table options are:

'connector'='kafka'
'format'='json'
'properties.bootstrap.servers'='kafka-service.kafka.svc.cluster.local:9092'
'properties.group.id'='nexmark'
'scan.startup.mode'='earliest-offset'
'sink.partitioner'='round-robin'
'topic'='event-demo'
	at org.apache.flink.table.factories.FactoryUtil.createDynamicTableSource(FactoryUtil.java:234)
	at org.apache.flink.table.planner.plan.schema.CatalogSourceTable.createDynamicTableSource(CatalogSourceTable.java:176)
	at org.apache.flink.table.planner.plan.schema.CatalogSourceTable.toRel(CatalogSourceTable.java:116)
	at org.apache.calcite.sql2rel.SqlToRelConverter.toRel(SqlToRelConverter.java:4060)
	at org.apache.calcite.sql2rel.SqlToRelConverter.convertIdentifier(SqlToRelConverter.java:2928)
	at org.apache.calcite.sql2rel.SqlToRelConverte